# LLM Forecasting Replication Experiment

This notebook replicates the core forecasting pipeline from the paper *Approaching Human-Level Forecasting with Language Models* (NeurIPS 2024), with the following modifications:
- GNews only (no NewsCatcher)
- Base model only (no fine-tuned model)
- 3 scratchpad prompts: NEW_1, NEW_2, NEW_3
- Mean aggregation across 3 prompt predictions

## 0. Configuration

In [1]:
# ── How many questions to evaluate ──
NUM_QUESTIONS = 1

# ── Model ──
#MODEL_NAME = "gpt-4.1-mini"
MODEL_NAME = "llama-3.3-70b-versatile"
TEMPERATURE = 1.0

# ── Retrieval ──
NUM_RETRIEVAL_DATES    = 5   # max retrieval dates per question
NUM_SEARCH_QUERIES     = 3   # search queries per source (GNews only)
NUM_ARTICLES_PER_QUERY = 10  # articles fetched per query
TOP_K_ARTICLES         = 15  # keep top-K after LLM relevance ranking
RELEVANCE_THRESHOLD    = 4   # 1-6 scale; keep articles rated >= this

# ── Output ──
import os
OUTPUT_DIR  = "../results/replication"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "results.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Imports & Setup

In [2]:
%load_ext autoreload
%autoreload 2

import sys, json, math, logging
from datetime import datetime, timedelta
import numpy as np

# ── make sure the llm_forecasting package is on the path ──
REPO_ROOT = os.path.abspath("../../llm_forecasting")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from config.constants import PROMPT_DICT, DEFAULT_RETRIEVAL_CONFIG
from prompts.base_reasoning import (
    BINARY_SCRATCH_PAD_PROMPT_NEW_1,
    BINARY_SCRATCH_PAD_PROMPT_NEW_2,
    BINARY_SCRATCH_PAD_PROMPT_NEW_3,
)
import ranking
import summarize
import model_eval
import information_retrieval
from utils import time_utils, string_utils

logging.basicConfig(level=logging.INFO)
print("Imports OK")

/opt/miniconda3/envs/myenv/lib/python3.11/site-packages/newspaper/parsers.py:19: UserWarning: nltk is not installed. Some NLP features will be unavailable. Install it with: pip install 'newspaper4k[nlp]'
  from . import text as txt
INFO:newspaper.network:Using requests library for http requests (alternative cloudscraper library is recommended for bypassing Cloudflare protection)
/opt/miniconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/sonia/Documents/llm_forecasting/llm_forecasting/model_eval.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai

Imports OK


## 2. Retrieval Config

In [3]:
RETRIEVAL_CONFIG = {
    **DEFAULT_RETRIEVAL_CONFIG,
    # model
    "SEARCH_QUERY_MODEL_NAME":   MODEL_NAME,
    "SUMMARIZATION_MODEL_NAME":  MODEL_NAME,
    "RANKING_MODEL_NAME":        MODEL_NAME,
    # retrieval
    "NUM_SEARCH_QUERY_KEYWORDS": NUM_SEARCH_QUERIES,
    "NUM_ARTICLES_PER_QUERY":    NUM_ARTICLES_PER_QUERY,
    "NUM_SUMMARIES_THRESHOLD":   TOP_K_ARTICLES,
    "RANKING_RELEVANCE_THRESHOLD": RELEVANCE_THRESHOLD,
    # prompts
    "SEARCH_QUERY_PROMPT_TEMPLATES": [
        PROMPT_DICT["search_query"]["0"],
        PROMPT_DICT["search_query"]["1"],
    ],
    "SUMMARIZATION_PROMPT_TEMPLATE": PROMPT_DICT["summarization"]["9"],
    "RANKING_PROMPT_TEMPLATE":       PROMPT_DICT["ranking"]["0"],
}

# The 3 reasoning prompts used for base model
REASONING_PROMPTS = [
    BINARY_SCRATCH_PAD_PROMPT_NEW_1,
    BINARY_SCRATCH_PAD_PROMPT_NEW_2,
    BINARY_SCRATCH_PAD_PROMPT_NEW_3,
]

## 3. Helper Functions

In [5]:
def get_retrieval_dates(date_begin, date_close, date_resolve, num_retrievals=5):
    """
    Return up to num_retrievals exponentially-spaced retrieval dates,
    using the same logic as time_utils.get_retrieval_date.
    Dates that fall after date_resolve are excluded.
    """
    dates = []
    for i in range(1, num_retrievals + 1):   # index starts at 1 (skip day 0)
        d = time_utils.get_retrieval_date(
            retrieval_index=i,
            num_retrievals=num_retrievals,
            date_begin=date_begin,
            date_close=date_close,
            resolve_date=date_resolve,
        )
        if d is not None:
            dates.append(d)
    return dates


def get_crowd_prediction_at_date(retrieval_date, community_predictions):
    """
    Return the crowd prediction closest to retrieval_date.
    community_predictions is a list of [date_str, probability] pairs.
    """
    preds = [p for p in community_predictions
             if time_utils.is_more_recent(p[0], retrieval_date, or_equal_to=True)]
    if not preds:
        preds = community_predictions   # fall back to all
    closest = time_utils.find_pred_with_closest_date(retrieval_date, preds)
    return closest[1] if closest else None


def brier_score(prediction, resolution):
    """Brier score for a single binary prediction."""
    return (prediction - resolution) ** 2


print("Helpers defined")

Helpers defined


## 4. Load Questions

In [6]:
DATA_PATH = "../../data/train.json"

with open(DATA_PATH) as f:
    all_questions = json.load(f)

# Select the first NUM_QUESTIONS questions
questions_to_run = all_questions[:NUM_QUESTIONS]
print(f"Loaded {len(all_questions)} questions. Running on {len(questions_to_run)}.")
print("\nFirst question:", questions_to_run[0]["question"])

Loaded 3762 questions. Running on 1.

First question: Will advanced LIGO announce discovery of gravitational waves by Jan. 31 2016?


## 5. Main Experiment Loop

In [7]:
all_results = []   # will be written to OUTPUT_FILE

for q_idx, q in enumerate(questions_to_run):
    print(f"\n{'='*60}")
    print(f"Question {q_idx+1}/{len(questions_to_run)}: {q['question']}")
    print('='*60)

    # ── Parse question fields ──────────────────────────────────
    question          = q["question"]
    background        = q["background"]
    resolution_criteria = q["resolution_criteria"]
    resolution        = float(q["resolution"])     # ground truth 0 or 1
    date_begin        = q["date_begin"]
    date_close        = q["date_close"]
    date_resolve      = q["date_resolve_at"]
    community_preds   = json.loads(q["community_predictions"])  # list of [date, prob]
    urls              = json.loads(q.get("extracted_urls", "[]"))

    # ── Compute retrieval dates ────────────────────────────────
    retrieval_dates = get_retrieval_dates(
        date_begin, date_close, date_resolve, NUM_RETRIEVAL_DATES
    )
    print(f"Retrieval dates ({len(retrieval_dates)}): {retrieval_dates}")

    question_results = {
        "question":            question,
        "resolution":          resolution,
        "date_begin":          date_begin,
        "date_close":          date_close,
        "retrieval_dates":     retrieval_dates,
        "retrieval_date_results": []
    }

    # ── Loop over retrieval dates ──────────────────────────────
    for r_idx, retrieval_date in enumerate(retrieval_dates):
        print(f"\n  -- Retrieval date {r_idx+1}/{len(retrieval_dates)}: {retrieval_date}")

        date_range = [date_begin, retrieval_date]

        # ── Step 1: Retrieve, rank, summarise articles ─────────
        try:
            (
                ranked_articles,
                all_articles,
                search_queries_gnews,
                _,   # search_queries_nc (empty — we pass [] for NC)
            ) = await ranking.retrieve_summarize_and_rank_articles(
                question,
                background,
                resolution_criteria,
                date_range,
                urls=urls,
                config=RETRIEVAL_CONFIG,
                return_intermediates=True,
            )
        except Exception as e:
            print(f"  [ERROR] Retrieval failed: {e}")
            question_results["retrieval_date_results"].append({
                "retrieval_date": retrieval_date,
                "error": str(e)
            })
            continue

        print(f"  Retrieved {len(all_articles)} articles, {len(ranked_articles)} after ranking.")

        # ── Step 2: Concatenate summaries ──────────────────────
        all_summaries = summarize.concat_summaries(
            ranked_articles[:RETRIEVAL_CONFIG["NUM_SUMMARIES_THRESHOLD"]]
        )

        # ── Step 3: Run 3 reasoning prompts, collect predictions
        today_to_close = [retrieval_date, date_close]
        base_reasonings   = []
        base_predictions  = []
        full_prompts      = []

        for p_idx, prompt_template in enumerate(REASONING_PROMPTS):
            print(f"  Running reasoning prompt {p_idx+1}/3 ...")
            try:
                reasonings, prompts = await model_eval.async_make_forecast(
                    question=question,
                    background_info=background,
                    resolution_criteria=resolution_criteria,
                    dates=today_to_close,
                    retrieved_info=all_summaries,
                    reasoning_prompt_templates=[prompt_template],
                    model_name=MODEL_NAME,
                    temperature=TEMPERATURE,
                    return_prompt=True,
                )
                reasoning = reasonings[0]
                prediction = string_utils.extract_prediction(reasoning, answer_type="probability")
                base_reasonings.append(reasoning)
                base_predictions.append(prediction)
                full_prompts.append(prompts[0])
                print(f"    Prediction: {prediction:.3f}")
            except Exception as e:
                print(f"    [ERROR] Reasoning prompt {p_idx+1} failed: {e}")
                base_reasonings.append(None)
                base_predictions.append(None)
                full_prompts.append(None)

        # ── Step 4: Average predictions ────────────────────────
        valid_preds = [p for p in base_predictions if p is not None]
        mean_prediction = float(np.mean(valid_preds)) if valid_preds else None
        print(f"  Mean prediction: {mean_prediction}")

        # ── Step 5: Crowd prediction & Brier scores ────────────
        crowd_prediction = get_crowd_prediction_at_date(retrieval_date, community_preds)
        llm_brier   = brier_score(mean_prediction, resolution) if mean_prediction is not None else None
        crowd_brier = brier_score(crowd_prediction, resolution) if crowd_prediction is not None else None
        print(f"  Crowd prediction: {crowd_prediction}  |  LLM Brier: {llm_brier:.4f}  Crowd Brier: {crowd_brier:.4f}")

        # ── Save everything for this retrieval date ────────────
        question_results["retrieval_date_results"].append({
            "retrieval_date":       retrieval_date,
            "num_articles_total":   len(all_articles),
            "num_articles_ranked":  len(ranked_articles),
            "search_queries_gnews": search_queries_gnews,
            "article_summaries":    all_summaries,
            "base_reasonings":      base_reasonings,
            "full_prompts":         full_prompts,
            "base_predictions":     base_predictions,
            "mean_prediction":      mean_prediction,
            "crowd_prediction":     crowd_prediction,
            "llm_brier_score":      llm_brier,
            "crowd_brier_score":    crowd_brier,
        })

    # ── Aggregate Brier scores across all retrieval dates ──────
    date_results = question_results["retrieval_date_results"]
    llm_briers   = [r["llm_brier_score"]   for r in date_results if r.get("llm_brier_score") is not None]
    crowd_briers = [r["crowd_brier_score"]  for r in date_results if r.get("crowd_brier_score") is not None]
    question_results["mean_llm_brier"]   = float(np.mean(llm_briers))   if llm_briers   else None
    question_results["mean_crowd_brier"] = float(np.mean(crowd_briers)) if crowd_briers else None

    print(f"\n  ► Mean LLM Brier:   {question_results['mean_llm_brier']}")
    print(f"  ► Mean Crowd Brier: {question_results['mean_crowd_brier']}")

    all_results.append(question_results)

    # ── Save incrementally after each question ─────────────────
    with open(OUTPUT_FILE, "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"  Results saved to {OUTPUT_FILE}")

print("\nAll done!")

INFO:ranking:Finding 3 search query keywords via LLM...



Question 1/1: Will advanced LIGO announce discovery of gravitational waves by Jan. 31 2016?
Retrieval dates (5): ['2015-10-04', '2015-10-07', '2015-10-15', '2015-11-02', '2015-12-14']

  -- Retrieval date 1/5: 2015-10-04


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:ranking:Search queries for NC: ['advanced LIGO observation run 2015', 'Will advanced LIGO announce discovery of gravitational waves by Jan. 31 2016?', 'advanced LIGO gravitational waves announcement', 'advanced LIGO status update', 'LIGO discovery timeline 2016', 'LIGO gravitational waves announcement timeline', 'advanced LIGO operational status']
INFO:ranking:Search queries for GNews: ['LIGO announcement timeline gravitational waves', 'Will advanced LIGO announce discovery of gravitational waves by Jan. 31 2016?', 'advanced LIGO operational status 2015', 'advanced LIGO observation runs schedule', 'gravitational waves detec

  Retrieved 0 articles, 0 after ranking.
  Running reasoning prompt 1/3 ...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:model_eval:Finished 1 base reasonings generated by llama-3.3-70b-versatile
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 27.000000 seconds


...

    Prediction: 0.300
  Running reasoning prompt 2/3 ...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:model_eval:Finished 1 base reasonings generated by llama-3.3-70b-versatile


    Prediction: 0.600
  Running reasoning prompt 3/3 ...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:model_eval:Finished 1 base reasonings generated by llama-3.3-70b-versatile


    Prediction: 0.600
  Mean prediction: 0.5
  Crowd prediction: 0.589  |  LLM Brier: 0.2500  Crowd Brier: 0.3469

  ► Mean LLM Brier:   0.24961111111111114
  ► Mean Crowd Brier: 0.4502641999999999
  Results saved to ../results/replication/results.json

All done!


## 6. Summary

In [9]:
print(f"{'Question':<60} {'LLM Brier':>10} {'Crowd Brier':>12}")
print("-" * 84)
for r in all_results:
    q_short = r["question"][:57] + "..." if len(r["question"]) > 60 else r["question"]
    llm_b   = f"{r['mean_llm_brier']:.4f}"   if r["mean_llm_brier"]   is not None else "N/A"
    crowd_b = f"{r['mean_crowd_brier']:.4f}" if r["mean_crowd_brier"] is not None else "N/A"
    print(f"{q_short:<60} {llm_b:>10} {crowd_b:>12}")

all_llm_briers   = [r["mean_llm_brier"]   for r in all_results if r["mean_llm_brier"]   is not None]
all_crowd_briers = [r["mean_crowd_brier"] for r in all_results if r["mean_crowd_brier"] is not None]
if all_llm_briers:
    print("-" * 84)
    print(f"{'OVERALL MEAN':<60} {np.mean(all_llm_briers):>10.4f} {np.mean(all_crowd_briers):>12.4f}")

Question                                                      LLM Brier  Crowd Brier
------------------------------------------------------------------------------------
Will advanced LIGO announce discovery of gravitational wa...     0.2496       0.4503
------------------------------------------------------------------------------------
OVERALL MEAN                                                     0.2496       0.4503
